# Part C — Additional 20-Year Variables ~ GO-i (K=2)
**Project:** Saving Brains — KMC 20-Year Follow-up  
**Module:** Part C v3 — Multi-domain adult functioning vs cognitive phenotype  

---
### Research question
Which dimensions of adult functioning at age 20 (education, behaviour,
attention, metabolic profile, vision, hearing, paediatrics, life habits,
follow-up) differ between cognitive phenotypes GO-1 High and GO-2 Low?

### Modules analysed (9 domains)
ICFES, ABCL, CONNERS, METP, OPHT, AUDIO, PEDIATRIC, LIFEHABITS, FOLLOW

### Statistical tests (K=2)
| Variable type | Test |
|---|---|
| Continuous | Mann-Whitney U (two-sided) + Spearman r (GO-i as ordinal 1→2) |
| Binary / Categorical | Pearson chi-square |

### Multiple comparison correction
FDR Benjamini-Hochberg applied **within each module separately**.  
q < 0.05 primary | q < 0.10 exploratory


## 1. Environment Setup & Imports

In [0]:
"""
Suppress threadpoolctl warnings common in Databricks multi-threaded env.
Must be set before any numpy/scipy imports.
"""
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import sys
import logging
import tempfile
import warnings
warnings.filterwarnings("ignore")

_stderr_orig = sys.stderr
sys.stderr = open(os.devnull, "w")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import mlflow

from scipy.stats import mannwhitneyu, chi2_contingency, spearmanr
from statsmodels.stats.multitest import multipletests

sys.stderr.close()
sys.stderr = _stderr_orig

print("Imports OK ✓")


## 2. Configuration

In [0]:
# ── Data paths ────────────────────────────────────────────────────────────
INPUT_MASTER   = "/Volumes/workspace/default/kmc_data/kmc_dataset_procesado_completo.csv"
INPUT_CLUSTERS = "/Volumes/workspace/default/kmc_data/clusters_GOiv5.csv"

# Output filenames (logged as MLflow artifacts; no permanent local writes)
OUT_FULL    = "partc_results_full.csv"
OUT_SIG     = "partc_results_significant.csv"
OUT_MISSING = "partc_missing_report.csv"
OUT_OUTLIER = "partc_outlier_report.csv"

# ── Analysis thresholds ────────────────────────────────────────────────────
FDR_ALPHA   = 0.05   # Primary FDR threshold
FDR_EXPLORE = 0.10   # Exploratory FDR threshold
MIN_N_GROUP = 5      # Minimum group size to include a variable
IQR_MULT    = 3.0    # Winsorization fence multiplier
WINSORIZE   = True   # Apply Winsorization to continuous variables

# ── MLflow experiment ─────────────────────────────────────────────────────
# In Databricks the tracking server is pre-configured; only set the path.
MLFLOW_EXPERIMENT = "/Users/your_user@email.com/saving_brains/part_c_vars20y"

# ── GO-i display settings ─────────────────────────────────────────────────
COLORS_GO = {1: "#2E86AB", 2: "#C73E1D"}
LABELS_GO = {1: "GO-1 High", 2: "GO-2 Low"}

# ── Module definitions (9 domains) ────────────────────────────────────────
MODULES = {
    "ICFES": {
        "prefixes":     ["ICFES_"],
        "description":  "Education and academic achievement",
        "note":         "ICFES = Colombian national exam. Scores ~50±10.",
    },
    "ABCL": {
        "prefixes":     ["ABCL_", "ASR_"],
        "description":  "Adult behaviour (ABCL/ASR)",
        "note":         "T-scales: 50=mean, 65=borderline, 70=clinical.",
    },
    "CONNERS": {
        "prefixes":     ["CONNERS_"],
        "description":  "Attention/ADHD symptoms (Conners)",
        "note":         "T-scales: higher = more symptoms.",
    },
    "METP": {
        "prefixes":     ["METP_", "PERFMET_"],
        "description":  "Metabolic profile",
        "note":         "Glucose, cholesterol, triglycerides, blood pressure.",
    },
    "OPHT": {
        "prefixes":     ["OPHT_"],
        "description":  "Ophthalmological evaluation",
        "note":         "Visual acuity, refraction, strabismus.",
    },
    "AUDIO": {
        "prefixes":     ["AUDIO_"],
        "description":  "Audiometric evaluation",
        "note":         "Hearing thresholds by frequency (dB).",
    },
    "PEDIATRIC": {
        "prefixes":     ["PEDIATRIC_"],
        "description":  "General paediatric assessment",
        "note":         "Height, weight, BMI, neurological development.",
    },
    "LIFEHABITS": {
        "prefixes":     ["LIFEHABITS_"],
        "description":  "Life habits",
        "note":         "Alcohol, tobacco, physical activity, sleep.",
    },
    "FOLLOW": {
        "prefixes":     ["FOLLOW_", "RT_"],
        "description":  "Clinical follow-up + Fragility",
        "note":         "Fragility Rasch, days with mother, days in ICU.",
    },
}

# Module color palette (for figures)
MODULE_COLORS = {
    "ICFES": "#2E86AB", "ABCL": "#F39C12", "CONNERS": "#C73E1D",
    "METP": "#6A0572", "OPHT": "#27AE60", "AUDIO": "#E67E22",
    "PEDIATRIC": "#8E44AD", "LIFEHABITS": "#16A085", "FOLLOW": "#2C3E50",
}


def shorten(col: str, max_len: int = 30) -> str:
    """Strip module prefixes and truncate column name for display."""
    prefixes = [
        "ICFES_", "ABCL_", "ASR_", "CONNERS_", "METP_",
        "OPHT_", "AUDIO_", "PEDIATRIC_", "LIFEHABITS_",
        "FOLLOW_", "RT_", "PERFMET_",
    ]
    s = col
    for p in prefixes:
        s = s.replace(p, "")
    return s[:max_len]


print("Configuration loaded ✓")


## 3. Logging & Utility Helpers

In [0]:
# Stream logs to stdout (Databricks captures cell output)
_fmt = logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s")
_ch  = logging.StreamHandler(sys.stdout)
_ch.setFormatter(_fmt)
logging.basicConfig(level=logging.INFO, handlers=[_ch], force=True)
logger = logging.getLogger("part_c_vars20y")


def to_num(series: pd.Series) -> pd.Series:
    """Coerce a Series to numeric; non-parseable values become NaN."""
    return pd.to_numeric(series, errors="coerce")


def var_type(series: pd.Series) -> str:
    """Classify a variable as continuous, binary, or categorical."""
    n = series.nunique()
    if n == 2:
        return "binary"
    if n <= 10:
        return "categorical"
    return "continuous"


def section(title: str = "", width: int = 76) -> None:
    """Log a labelled section separator."""
    t = title.strip()
    if t:
        dashes = "─" * max(2, width - len(t) - 6)
        logger.info("\n─── " + t + " " + dashes)
    else:
        logger.info("─" * width)


print("Helpers defined ✓")


## 4. Step 0 — Data Loading & Module Inventory

Merges the master dataset with GO-i labels and builds the column
inventory for each module — retaining only variables with at least
`MIN_N_GROUP` valid observations in **both** GO-i groups.

In [0]:
def load_data(master_path: str, clusters_path: str):
    """
    Load master dataset, merge GO-i labels, and build per-module column lists.

    Inclusion criteria per variable:
    - At least 2 unique values (excludes constants)
    - At least MIN_N_GROUP valid observations in each GO-i group

    Returns
    -------
    df       : merged DataFrame
    cols_mod : dict {module_name: [column_list]}
    """
    section("STEP 0 — DATA LOADING & MODULE INVENTORY")

    df       = pd.read_csv(master_path, low_memory=False)
    clusters = pd.read_csv(clusters_path)
    df       = df.merge(clusters[["code", "GO_i", "GO_i_label"]], on="code", how="inner")
    logger.info("Dataset: %d rows x %d cols", df.shape[0], df.shape[1])

    go = df["GO_i"]
    for g, label in LABELS_GO.items():
        logger.info("  %-14s: n=%d", label, (go == g).sum())

    cols_mod = {}
    for mod, cfg in MODULES.items():
        candidates = [
            c for pref in cfg["prefixes"]
            for c in df.columns if c.startswith(pref)
        ]
        cols_ok = []
        for col in candidates:
            s = to_num(df[col])
            if s.nunique() < 2:
                continue
            group_sizes = [s[go == g].dropna().shape[0] for g in [1, 2]]
            if all(n >= MIN_N_GROUP for n in group_sizes):
                cols_ok.append(col)
        cols_mod[mod] = cols_ok
        avg_complete = (
            df[cols_ok].apply(to_num).notna().mean().mean() * 100
            if cols_ok else 0.0
        )
        logger.info("  %-12s: %4d vars  %.0f%% avg completeness", mod, len(cols_ok), avg_complete)

    total_vars = sum(len(v) for v in cols_mod.values())
    logger.info("\n  Total variables to analyse: %d", total_vars)
    return df, cols_mod


df, cols_mod = load_data(INPUT_MASTER, INPUT_CLUSTERS)


## 5. Step 1 — Missingness Audit

Each of the 9 modules has distinct missing patterns driven by different
evaluation protocols. This step:
1. Quantifies missing % per module and per GO-i group
2. Classifies the mechanism via chi-square MCAR test
3. Flags MAR variables (missing depends on GO-i) — these are included
   but interpreted with caution

In [0]:
def audit_missing(df: pd.DataFrame, cols_mod: dict) -> pd.DataFrame:
    """
    Audit missingness for all module variables.

    MCAR test: chi-square between missingness indicator and GO-i membership.
    p < 0.05 → MAR (missing depends on cognitive phenotype).

    Returns
    -------
    df_miss : one row per variable
    """
    section("STEP 1 — MISSINGNESS AUDIT")
    go = df["GO_i"]

    logger.info(
        "  %-12s %-40s %5s %7s %7s %7s %6s",
        "Module", "Variable", "n", "%miss", "GO1%", "GO2%", "Mech",
    )
    logger.info("  " + "-" * 90)

    records = []
    for mod, cols in cols_mod.items():
        for col in cols:
            s       = to_num(df[col])
            n_total = len(s)
            n_miss  = int(s.isna().sum())
            pct     = n_miss / n_total * 100
            miss_g1 = s[go == 1].isna().mean() * 100
            miss_g2 = s[go == 2].isna().mean() * 100

            # MCAR test: does missingness differ by GO-i?
            p_mcar = np.nan
            if 0 < n_miss < n_total:
                ct = pd.crosstab(go, s.notna().astype(int))
                if ct.shape == (2, 2):
                    try:
                        _, p_mcar, _, _ = chi2_contingency(ct)
                    except Exception:
                        pass

            mechanism = (
                "MAR"  if not np.isnan(p_mcar) and p_mcar < 0.05
                else "MCAR" if not np.isnan(p_mcar)
                else "unknown"
            )
            if n_miss > 0:
                flag = "!" if mechanism == "MAR" else " "
                logger.info(
                    "  %s %-11s %-40s %5d %6.1f%% %6.1f%% %6.1f%% %6s",
                    flag, mod, col, n_total, pct, miss_g1, miss_g2, mechanism,
                )
            records.append({
                "module":       mod,
                "variable":     col,
                "n_total":      n_total,
                "n_missing":    n_miss,
                "pct_missing":  round(pct, 2),
                "miss_go1_pct": round(miss_g1, 2),
                "miss_go2_pct": round(miss_g2, 2),
                "p_mcar_chi2":  round(float(p_mcar), 4) if not np.isnan(p_mcar) else np.nan,
                "mechanism":    mechanism,
                "alert":        "MAR" if mechanism == "MAR" else "OK",
            })

    df_miss = pd.DataFrame(records)

    section("Missingness summary by module")
    logger.info(
        "  %-12s %10s %12s %14s %12s",
        "Module", "Total vars", "With missing", "MAR detected", "Avg pct miss",
    )
    logger.info("  " + "-" * 65)
    for mod in cols_mod:
        sub_m   = df_miss[df_miss["module"] == mod]
        n_miss  = (sub_m["n_missing"] > 0).sum()
        n_mar   = (sub_m["mechanism"] == "MAR").sum()
        pct_avg = sub_m["pct_missing"].mean()
        logger.info("  %-12s %10d %12d %14d %11.1f%%", mod, len(sub_m), n_miss, n_mar, pct_avg)

    n_mar_total = (df_miss["mechanism"] == "MAR").sum()
    logger.info("\n  Variables with MAR (p<0.05): %d", n_mar_total)
    if n_mar_total > 0:
        logger.info("  MAR variables are included but flagged — interpret with caution.")
        mar_df = df_miss[df_miss["mechanism"] == "MAR"]
        for _, r in mar_df.head(10).iterrows():
            logger.info(
                "    %-12s %-40s GO1=%.1f%%  GO2=%.1f%%",
                r["module"], r["variable"], r["miss_go1_pct"], r["miss_go2_pct"],
            )
    return df_miss


df_miss = audit_missing(df, cols_mod)


### Figure 1 — Missingness Heatmap by Module & GO-i

In [0]:
def plot_missing(df_miss: pd.DataFrame) -> plt.Figure:
    """
    Two-panel figure: module-level missingness heatmap and MAR bar chart.
    Displayed inline; returns Figure object for MLflow logging.
    """
    df_plot = df_miss[df_miss["n_missing"] > 0].copy()
    if df_plot.empty:
        print("No missing values in any module ✓")
        return None

    miss_mod = (
        df_plot.groupby("module")[["miss_go1_pct", "miss_go2_pct", "pct_missing"]]
        .mean().round(1)
    )
    miss_mod.columns = ["GO-1 High", "GO-2 Low", "Total"]
    miss_mod = miss_mod.sort_values("Total", ascending=False)

    n_mar_mod = (
        df_miss.groupby("module")["mechanism"]
        .apply(lambda x: (x == "MAR").sum())
        .reset_index()
    )
    n_mar_mod.columns = ["module", "n_MAR"]
    n_mar_mod = n_mar_mod[n_mar_mod["n_MAR"] > 0].sort_values("n_MAR", ascending=False)

    h = max(3, len(miss_mod) * 0.6 + 2)
    fig, axes = plt.subplots(1, 2, figsize=(14, h))

    sns.heatmap(
        miss_mod, cmap="Reds", vmin=0, vmax=100,
        annot=True, fmt=".1f", linewidths=0.4, ax=axes[0],
        cbar_kws={"label": "% missing", "shrink": 0.7},
        annot_kws={"size": 9},
    )
    axes[0].set_title("Average % missing by module and GO-i",
                      fontweight="bold", fontsize=11)
    axes[0].tick_params(axis="y", rotation=0, labelsize=9)

    if not n_mar_mod.empty:
        axes[1].barh(n_mar_mod["module"], n_mar_mod["n_MAR"],
                     color="#C73E1D", alpha=0.82, edgecolor="white")
        axes[1].set_xlabel("N variables with MAR (p<0.05)", fontsize=10)
        axes[1].set_title("Variables with differential missingness\nby GO-i (MAR)",
                          fontweight="bold", fontsize=11)
        axes[1].invert_yaxis()
    else:
        axes[1].text(0.5, 0.5, "No MAR detected ✓",
                     ha="center", va="center", fontsize=13,
                     transform=axes[1].transAxes, color="green")
        axes[1].axis("off")

    plt.suptitle(
        "Missingness Analysis — Part C (K=2)\n"
        "MAR = differential missing between GO-1 and GO-2 (chi-sq p<0.05)",
        fontweight="bold", fontsize=12,
    )
    plt.tight_layout()
    plt.show()  # Inline only — no file saved
    return fig


fig_missing = plot_missing(df_miss)


## 6. Step 2 — Outlier Audit & Winsorization

Strategy by variable type:
- **Continuous** → Winsorization ±3×IQR (if IQR > 0)
- **IQR = 0** → no treatment (recorded as `N/A_concentrated`)
- **Binary / Categorical** → no treatment

> Mann-Whitney is rank-based and robust to outliers by design.
> Winsorization affects reported group means and boxplot visualisations.

In [0]:
def _winsorize_col(series: pd.Series, name: str, iqr_mult: float = IQR_MULT):
    """
    Detect and optionally Winsorize a single continuous Series.

    Falls back to P2.5-P97.5 range when IQR=0.
    Returns (summary_dict, cleaned_series).
    """
    n_valid = series.notna().sum()
    if n_valid < 10:
        return None, series

    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr    = q3 - q1

    if iqr == 0:
        return {
            "variable": name, "n_outliers": 0, "pct_outliers": 0.0,
            "alert": "N/A_concentrated", "winsorized": False,
            "note": "IQR=0: near-constant distribution",
        }, series

    lo, hi = q1 - iqr_mult * iqr, q3 + iqr_mult * iqr
    mask   = ((series < lo) | (series > hi)) & series.notna()
    n_out  = int(mask.sum())
    pct    = n_out / n_valid * 100

    s_clean = series.clip(lower=lo, upper=hi) if WINSORIZE and n_out > 0 else series.copy()

    return {
        "variable":    name,
        "n_valid":     int(n_valid),
        "mean":        round(float(series.mean()), 3),
        "sd":          round(float(series.std()), 3),
        "Q1":          round(float(q1), 3),
        "Q3":          round(float(q3), 3),
        "IQR":         round(float(iqr), 3),
        "lower_bound": round(float(lo), 3),
        "upper_bound": round(float(hi), 3),
        "n_outliers":  n_out,
        "pct_outliers": round(pct, 2),
        "alert":       "HIGH" if pct > 5 else "MEDIUM" if pct > 1 else "OK",
        "winsorized":  WINSORIZE and n_out > 0,
        "note":        "",
    }, s_clean


def audit_outliers(df: pd.DataFrame, cols_mod: dict):
    """
    Audit and Winsorize all continuous module variables.

    Returns
    -------
    df_clean    : DataFrame with Winsorized continuous columns
    df_report   : outlier audit summary (one row per variable)
    """
    iqr_label = str(IQR_MULT)
    section("STEP 2 — OUTLIER AUDIT (+-" + iqr_label + "xIQR | continuous only)")
    win_label = str(WINSORIZE)
    logger.info("  Winsorize: %s", win_label)

    df_clean = df.copy()
    records  = []
    flagged  = []

    for mod, cols in cols_mod.items():
        for col in cols:
            s    = to_num(df[col])
            vtyp = var_type(s)

            if vtyp != "continuous":
                records.append({
                    "module": mod, "variable": col, "var_type": vtyp,
                    "n_outliers": 0, "pct_outliers": 0.0,
                    "alert": "N/A_categorical", "winsorized": False,
                })
                continue

            res, s_clean = _winsorize_col(s, col)
            if res is None:
                continue
            res["module"]   = mod
            res["var_type"] = vtyp
            records.append(res)

            if res["n_outliers"] > 0:
                df_clean[col] = s_clean
                icon = "RED" if res["alert"] == "HIGH" else "YEL" if res["alert"] == "MEDIUM" else "OK"
                flagged.append((mod, col, res["n_outliers"], res["pct_outliers"], res["alert"], icon))

    if flagged:
        logger.info("  %-12s %-40s %6s %6s  Alert", "Module", "Variable", "n_out", "pct")
        logger.info("  " + "-" * 72)
        for mod, col, n_out, pct_out, alert, icon in sorted(flagged, key=lambda x: -x[3]):
            logger.info("  [%s] %-11s %-40s %6d %5.1f%%  %s", icon, mod, col, n_out, pct_out, alert)

    df_report = pd.DataFrame(records)

    section("Outlier summary by module")
    logger.info(
        "  %-12s %10s %12s %10s %12s",
        "Module", "Continuous", "With outlier", "Total out", "Alert HIGH",
    )
    logger.info("  " + "-" * 62)
    for mod in cols_mod:
        sub_m  = df_report[(df_report["module"] == mod) & (df_report["var_type"] == "continuous")]
        n_con  = int((sub_m["n_outliers"] > 0).sum()) if not sub_m.empty else 0
        total  = int(sub_m["n_outliers"].sum()) if not sub_m.empty else 0
        n_high = int((sub_m["alert"] == "HIGH").sum()) if not sub_m.empty else 0
        logger.info("  %-12s %10d %12d %10d %12d", mod, len(sub_m), n_con, total, n_high)

    n_win = int((df_report["winsorized"] == True).sum())
    logger.info("\n  Variables Winsorized: %d", n_win)
    logger.info("  Variables unchanged : %d", len(df_report) - n_win)

    return df_clean, df_report


df_clean, df_outlier_report = audit_outliers(df, cols_mod)
print("\n✓ Clean dataset ready — use df_clean in statistical analysis.")


### Figure 2 — Outlier Distribution by Module

In [0]:
def plot_outliers(df_outlier_report: pd.DataFrame) -> plt.Figure:
    """
    Two-panel outlier summary: top-20 variable bar chart and module summary.
    Displayed inline; returns Figure object for MLflow logging.
    """
    df_cont = df_outlier_report[
        (df_outlier_report["var_type"] == "continuous") &
        (df_outlier_report["n_outliers"] > 0)
    ].copy()

    if df_cont.empty:
        print("No outliers detected in continuous variables ✓")
        return None

    df_top = df_cont.nlargest(20, "pct_outliers")
    bar_colors = [MODULE_COLORS.get(r["module"], "#888") for _, r in df_top.iterrows()]
    labels_top = df_top["variable"].str[:35]

    h = max(4, len(df_top) * 0.35 + 2)
    fig, axes = plt.subplots(1, 2, figsize=(14, h))

    # Panel A: top-20 variables by outlier %
    axes[0].barh(labels_top, df_top["pct_outliers"], color=bar_colors,
                 alpha=0.82, edgecolor="white")
    axes[0].axvline(1.0, color="orange", ls="--", lw=1.5, label="1% (MEDIUM)")
    axes[0].axvline(5.0, color="red",    ls="--", lw=1.5, label="5% (HIGH)")
    axes[0].set_xlabel("% participants flagged as outliers", fontsize=10)
    iqr_label = str(IQR_MULT)
    axes[0].set_title("Top-20 variables with most outliers\n(+-" + iqr_label + "xIQR)",
                      fontweight="bold", fontsize=11)
    legend_patches = [
        plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.82)
        for c in MODULE_COLORS.values()
    ]
    axes[0].legend(
        legend_patches
        + [plt.Line2D([0], [0], color="orange", ls="--", lw=1.5),
           plt.Line2D([0], [0], color="red",    ls="--", lw=1.5)],
        list(MODULE_COLORS.keys()) + ["1% threshold", "5% threshold"],
        fontsize=7, loc="lower right", ncol=2,
    )
    axes[0].invert_yaxis()

    # Panel B: module-level summary
    mod_summary = (
        df_cont.groupby("module")
        .agg(n_vars=("n_outliers", lambda x: (x > 0).sum()),
             total_out=("n_outliers", "sum"),
             pct_mean=("pct_outliers", "mean"))
        .reset_index()
        .sort_values("total_out", ascending=False)
    )
    x = np.arange(len(mod_summary))
    w = 0.4
    axes[1].bar(x - w / 2, mod_summary["n_vars"], w,
                label="N vars with outlier", color="#2E86AB", alpha=0.82)
    axes[1].bar(x + w / 2, mod_summary["pct_mean"], w,
                label="Avg outlier %", color="#F39C12", alpha=0.82)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(mod_summary["module"], rotation=30, ha="right", fontsize=9)
    axes[1].set_ylabel("N vars / avg pct", fontsize=10)
    axes[1].set_title("Outlier summary by module", fontweight="bold", fontsize=11)
    axes[1].legend(fontsize=9)

    win_label = str(WINSORIZE)
    plt.suptitle(
        "Outlier Audit — Part C (K=2)  |  Winsorize=" + win_label + "  |  Threshold=+-" + iqr_label + "xIQR",
        fontweight="bold", fontsize=12,
    )
    plt.tight_layout()
    plt.show()  # Inline only — no file saved
    return fig


fig_outliers = plot_outliers(df_outlier_report)


## 7. Step 3 — Statistical Analysis by Module (K=2)

For each module:
- **Continuous** → Mann-Whitney U (two-sided) + Spearman r
- **Binary / Categorical** → Pearson chi-square
- FDR Benjamini-Hochberg correction applied **within each module**

In [0]:
def _analyse_module(df: pd.DataFrame, cols: list, mod: str, go: pd.Series) -> pd.DataFrame:
    """
    Run statistical tests for all variables in a single module.

    Returns
    -------
    DataFrame with one row per variable containing test statistics,
    group means, and delta % (GO-2 vs GO-1).
    """
    rows = []
    for col in cols:
        s    = to_num(df[col])
        vtyp = var_type(s)
        n    = int(s.notna().sum())
        g1   = s[go == 1].dropna().values
        g2   = s[go == 2].dropna().values

        m1  = float(np.mean(g1)) if len(g1) else np.nan
        m2  = float(np.mean(g2)) if len(g2) else np.nan
        sd1 = float(np.std(g1))  if len(g1) else np.nan
        sd2 = float(np.std(g2))  if len(g2) else np.nan

        stat, p_raw = np.nan, np.nan
        r_sp, p_sp  = np.nan, np.nan

        try:
            if vtyp == "continuous":
                stat, p_raw = mannwhitneyu(g1, g2, alternative="two-sided")
                ok = s.notna()
                r_sp, p_sp = spearmanr(s[ok], go[ok])
            else:
                ct = pd.crosstab(go, s)
                if ct.shape[0] >= 2 and ct.shape[1] >= 2:
                    stat, p_raw, _, _ = chi2_contingency(ct)
        except Exception:
            pass

        delta_pct = (
            (m2 - m1) / abs(m1) * 100
            if vtyp == "continuous" and not np.isnan(m1) and m1 != 0
            else np.nan
        )

        rows.append({
            "module":        mod,
            "variable":      col,
            "label":         shorten(col),
            "var_type":      vtyp,
            "n":             n,
            "n_go1":         len(g1),
            "n_go2":         len(g2),
            "stat":          round(float(stat), 3) if not np.isnan(stat) else np.nan,
            "p_raw":         round(float(p_raw), 6) if not np.isnan(p_raw) else np.nan,
            "r_spearman":    round(float(r_sp), 3)  if not np.isnan(r_sp)  else np.nan,
            "p_spearman":    round(float(p_sp), 4)  if not np.isnan(p_sp)  else np.nan,
            "mean_go1":      round(m1, 3)  if not np.isnan(m1)  else np.nan,
            "sd_go1":        round(sd1, 3) if not np.isnan(sd1) else np.nan,
            "mean_go2":      round(m2, 3)  if not np.isnan(m2)  else np.nan,
            "sd_go2":        round(sd2, 3) if not np.isnan(sd2) else np.nan,
            "delta_pct_go2_vs_go1": round(delta_pct, 1) if not np.isnan(delta_pct) else np.nan,
        })
    return pd.DataFrame(rows)


def run_analysis(df: pd.DataFrame, cols_mod: dict):
    """
    Run statistical analysis across all modules with per-module FDR correction.

    Returns
    -------
    df_all  : concatenated results for all modules
    dfs_mod : dict {module_name: per-module DataFrame}
    """
    section("STEP 3 — STATISTICAL ANALYSIS BY MODULE (K=2)")
    go      = df["GO_i"]
    dfs_mod = {}

    for mod, cols in cols_mod.items():
        if not cols:
            continue
        desc = MODULES[mod]["description"]
        note = MODULES[mod]["note"]
        section("Module " + mod + " — " + desc)
        logger.info("  %s", note)

        df_mod = _analyse_module(df, cols, mod, go)

        # FDR correction within module
        mask_p = df_mod["p_raw"].notna()
        ps     = df_mod.loc[mask_p, "p_raw"].values
        if len(ps) > 0:
            _, qs, _, _ = multipletests(ps, method="fdr_bh")
            df_mod.loc[mask_p, "q_fdr"] = qs.round(6)
        else:
            df_mod["q_fdr"] = np.nan

        df_mod = df_mod.sort_values("p_raw")
        n_sig  = int((df_mod["q_fdr"] < FDR_ALPHA).sum())
        n_exp  = int((df_mod["q_fdr"] < FDR_EXPLORE).sum())
        logger.info("\n  q<0.05: %d  |  q<0.10: %d  |  total: %d", n_sig, n_exp, len(df_mod))

        sig = df_mod[df_mod["q_fdr"] < FDR_EXPLORE]
        if not sig.empty:
            logger.info(
                "\n  %-40s %10s %7s %9s %9s  GO1     GO2     delta%%",
                "Variable", "var_type", "r_sp", "p_raw", "q_fdr",
            )
            logger.info("  " + "-" * 100)
            for _, r in sig.iterrows():
                mark = (
                    "***" if r["q_fdr"] < 0.001
                    else "**"  if r["q_fdr"] < 0.01
                    else "*"   if r["q_fdr"] < 0.05
                    else "†"
                )
                r_sp_str    = "{:.3f}".format(r["r_spearman"]) if not np.isnan(r["r_spearman"]) else ""
                delta_str   = "{:.1f}".format(r["delta_pct_go2_vs_go1"]) if not np.isnan(r["delta_pct_go2_vs_go1"]) else ""
                mean_g1_str = "{:.2f}".format(r["mean_go1"]) if not np.isnan(r["mean_go1"]) else ""
                mean_g2_str = "{:.2f}".format(r["mean_go2"]) if not np.isnan(r["mean_go2"]) else ""
                logger.info(
                    "  %-40s %10s %7s %9.5f %9.5f  %-6s  %-6s  %-5s  %s",
                    r["label"], r["var_type"], r_sp_str,
                    r["p_raw"], r["q_fdr"],
                    mean_g1_str, mean_g2_str, delta_str, mark,
                )
        else:
            logger.info("  No significant variables (q<0.10) in this module")
            for _, r in df_mod.head(3).iterrows():
                logger.info("  (top p_raw) %-38s p=%.4f", r["label"], r["p_raw"])

        dfs_mod[mod] = df_mod

    df_all = pd.concat(dfs_mod.values(), ignore_index=True)
    logger.info("\n  Analysis complete — total rows: %d", len(df_all))
    return df_all, dfs_mod


# Use df_clean (Winsorized) for statistical analysis
df_all, dfs_mod = run_analysis(df_clean, cols_mod)


## 8. Step 4 — Results Figures

All figures displayed **inline only** — saved to a temp directory and
logged as MLflow artifacts in Step 5.

| Figure | Description |
|---|---|
| Fig 3 | ICFES — boxplots (continuous) and bar charts (binary) |
| Fig 4a/b | ABCL and CONNERS — boxplots |
| Fig 5a/b/c | METP, LIFEHABITS, FOLLOW — boxplots |
| Fig 6 | Heatmap — % difference GO-2 vs GO-1 (q<0.05 continuous) |
| Fig 7 | Forest plot — all significant structures sorted by Δ% |

In [0]:
def _boxplots(df, var_list, go, title, n_cols=3):
    """
    Boxplot grid for continuous variables (K=2: GO-1 vs GO-2).
    Returns Figure object (inline display).
    """
    cont_vars = [v for v in var_list if to_num(df[v]).nunique() > 10]
    if not cont_vars:
        return None

    n_rows = max(1, (len(cont_vars) + n_cols - 1) // n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 3.5 * n_rows))
    axes_flat = np.array(axes).flatten()

    for i, col in enumerate(cont_vars):
        ax = axes_flat[i]
        s  = to_num(df[col])
        data_groups = [s[go == g].dropna().values for g in [1, 2]]
        bp = ax.boxplot(
            data_groups, patch_artist=True, widths=0.55,
            medianprops=dict(color="white", linewidth=2),
        )
        for patch, g in zip(bp["boxes"], [1, 2]):
            patch.set_facecolor(COLORS_GO[g])
            patch.set_alpha(0.82)
        tick_labels = [
            LABELS_GO[g] + "\n(n=" + str(len(data_groups[gi])) + ")"
            for gi, g in enumerate([1, 2])
        ]
        ax.set_xticklabels(tick_labels, fontsize=8)
        ax.set_title(shorten(col, 28), fontweight="bold", fontsize=9)

    for j in range(len(cont_vars), len(axes_flat)):
        fig.delaxes(axes_flat[j])

    plt.suptitle(title, fontweight="bold", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()  # Inline only — no file saved
    return fig


def _barcharts(df, var_list, go, title):
    """
    Grouped bar chart for binary/categorical variables (K=2).
    Returns Figure object (inline display).
    """
    cat_vars = [v for v in var_list if to_num(df[v]).nunique() <= 10]
    if not cat_vars:
        return None

    n_vars   = min(len(cat_vars), 12)
    cat_vars = cat_vars[:n_vars]
    x = np.arange(n_vars)
    w = 0.35

    fig, ax = plt.subplots(figsize=(max(10, n_vars * 1.2), 5))
    for gi, g in enumerate([1, 2]):
        vals = []
        for col in cat_vars:
            s = to_num(df[col])
            v = s[go == g].dropna()
            if v.nunique() == 2:
                vals.append(v.mean())
            elif len(v) > 0:
                vals.append(v.value_counts(normalize=True).iloc[0])
            else:
                vals.append(np.nan)
        ax.bar(x + gi * w - w / 2, vals, w,
               color=COLORS_GO[g], alpha=0.82,
               label=LABELS_GO[g], edgecolor="white")

    ax.set_xticks(x)
    ax.set_xticklabels([shorten(v, 22) for v in cat_vars],
                       rotation=40, ha="right", fontsize=8)
    ax.set_ylabel("Proportion / mean", fontsize=10)
    ax.set_title(title, fontweight="bold", fontsize=11)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()  # Inline only — no file saved
    return fig


def plot_results(df_clean, dfs_mod, df_all):
    """
    Generate all results figures (inline).

    Returns
    -------
    list of (Figure, filename) tuples for MLflow logging
    """
    section("STEP 4 — RESULTS FIGURES")
    go   = df_clean["GO_i"]
    sns.set_theme(style="whitegrid")
    fig_list = []  # [(fig, filename), ...]

    # ── Fig 3: ICFES ─────────────────────────────────────────────────
    df_icfes = dfs_mod.get("ICFES", pd.DataFrame())
    if not df_icfes.empty:
        sig_vars = df_icfes[df_icfes["q_fdr"] < FDR_EXPLORE]["variable"].tolist()
        cont_sig = [v for v in sig_vars if to_num(df_clean[v]).nunique() > 10][:12]
        bin_sig  = [v for v in sig_vars if to_num(df_clean[v]).nunique() <= 2][:8]
        if cont_sig:
            f = _boxplots(df_clean, cont_sig, go,
                          "ICFES — Education & academic achievement by GO-i\n(continuous, q<0.10)",
                          n_cols=4)
            if f:
                fig_list.append((f, "fig_03a_icfes_continuous.png"))
        if bin_sig:
            f = _barcharts(df_clean, bin_sig, go,
                           "ICFES — Binary variables (proportion) by GO-i")
            if f:
                fig_list.append((f, "fig_03b_icfes_binary.png"))

    # ── Fig 4: ABCL + CONNERS ─────────────────────────────────────────
    for mod, fname, title in [
        ("ABCL",    "fig_04a_abcl.png",
         "Adult behaviour (ABCL/ASR) by GO-i\n(T-scales: >65 borderline, >70 clinical)"),
        ("CONNERS", "fig_04b_conners.png",
         "Attention/ADHD symptoms (Conners) by GO-i\n(T-scales: higher = more symptoms)"),
    ]:
        df_mod = dfs_mod.get(mod, pd.DataFrame())
        if df_mod.empty:
            continue
        top_vars = df_mod.sort_values("p_raw")["variable"].tolist()[:12]
        f = _boxplots(df_clean, top_vars, go, title, n_cols=4)
        if f:
            fig_list.append((f, fname))

    # ── Fig 5: METP, LIFEHABITS, FOLLOW ──────────────────────────────
    for mod, fname, title in [
        ("METP",       "fig_05a_metp.png",       "Metabolic profile by GO-i"),
        ("LIFEHABITS", "fig_05b_lifehabits.png", "Life habits by GO-i"),
        ("FOLLOW",     "fig_05c_follow.png",     "Clinical follow-up + Fragility by GO-i"),
    ]:
        df_mod = dfs_mod.get(mod, pd.DataFrame())
        if df_mod.empty:
            continue
        top_vars = df_mod.sort_values("p_raw")["variable"].tolist()[:12]
        f = _boxplots(df_clean, top_vars, go, title, n_cols=4)
        if f:
            fig_list.append((f, fname))

    # ── Fig 6: % difference heatmap ───────────────────────────────────
    # Z-score per column is uninformative with K=2 (always ±1).
    # % difference GO-2 vs GO-1 is more meaningful.
    df_sig = df_all[
        (df_all["q_fdr"] < FDR_ALPHA) &
        (df_all["var_type"] == "continuous")
    ].sort_values(["module", "q_fdr"])

    if len(df_sig) >= 3:
        top_sig  = df_sig.head(40)
        deltas   = []
        q_vals   = []
        xlabels  = []
        for _, r in top_sig.iterrows():
            m1 = r["mean_go1"]
            m2 = r["mean_go2"]
            d  = (m2 - m1) / abs(m1) * 100 if (not np.isnan(m1) and m1 != 0) else 0.0
            deltas.append(d)
            q_vals.append(r["q_fdr"])
            xlabels.append("[" + r["module"] + "] " + shorten(r["variable"], 24))

        annots = [
            "{:+.1f}%\nq={:.3f}".format(d, q)
            for d, q in zip(deltas, q_vals)
        ]
        fig_w = max(14, len(xlabels) * 0.9)
        fig6, ax = plt.subplots(figsize=(fig_w, 3.5))
        sns.heatmap(
            np.array(deltas).reshape(1, -1),
            cmap="RdBu_r", center=0, vmin=-20, vmax=20,
            annot=np.array(annots).reshape(1, -1), fmt="",
            linewidths=0.4, ax=ax,
            xticklabels=xlabels,
            yticklabels=["GO-2 Low vs GO-1 High"],
            cbar_kws={"label": "% diff (GO-2 - GO-1) / |GO-1|", "shrink": 0.5},
            annot_kws={"size": 7},
        )
        ax.set_title(
            "20-year additional variables — % difference GO-2 Low vs GO-1 High\n"
            "(continuous, q_FDR<0.05, sorted by module and q)\n"
            "red = GO-1 larger  |  blue = GO-2 larger",
            fontweight="bold", fontsize=11,
        )
        ax.tick_params(axis="x", rotation=45, labelsize=7.5)
        ax.tick_params(axis="y", labelsize=10, rotation=0)
        plt.tight_layout()
        plt.show()  # Inline only — no file saved
        fig_list.append((fig6, "fig_06_heatmap_significant.png"))

    # ── Fig 7: Forest plot ────────────────────────────────────────────
    if len(df_sig) >= 1:
        df_forest = df_sig.copy()
        df_forest["delta"] = df_forest.apply(
            lambda r: (r["mean_go2"] - r["mean_go1"]) / abs(r["mean_go1"]) * 100
            if r["mean_go1"] != 0 else 0.0,
            axis=1,
        )
        df_forest["plot_label"] = df_forest.apply(
            lambda r: "[" + r["module"] + "] " + shorten(r["variable"], 28),
            axis=1,
        )
        df_forest = df_forest.sort_values("delta")

        bar_colors = ["#2E86AB" if d > 0 else "#C73E1D" for d in df_forest["delta"]]
        h = max(5, len(df_forest) * 0.38)
        fig7, ax = plt.subplots(figsize=(10, h))
        ax.barh(df_forest["plot_label"], df_forest["delta"],
                color=bar_colors, alpha=0.82, edgecolor="white")
        ax.axvline(0, color="black", lw=1, ls="--", alpha=0.6)

        for i, (_, row) in enumerate(df_forest.iterrows()):
            mark = (
                "***" if row["q_fdr"] < 0.001
                else "**"  if row["q_fdr"] < 0.01
                else "*"   if row["q_fdr"] < 0.05
                else "†"
            )
            x_pos  = row["delta"] + (0.5 if row["delta"] >= 0 else -0.5)
            h_align = "left" if row["delta"] >= 0 else "right"
            ax.text(x_pos, i, mark, va="center", ha=h_align, fontsize=8, color="black")

        ax.set_xlabel("% difference GO-2 Low vs GO-1 High", fontsize=11)
        ax.set_title(
            "Forest plot — Significant variables (q<0.05)\n"
            "blue = GO-2 larger  |  red = GO-1 larger  |  *** q<0.001",
            fontweight="bold", fontsize=11,
        )
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()  # Inline only — no file saved
        fig_list.append((fig7, "fig_07_forest_significant.png"))

    return fig_list


fig_list = plot_results(df_clean, dfs_mod, df_all)


## 9. Step 5 — Executive Summary

In [0]:
def executive_summary(df_all: pd.DataFrame, dfs_mod: dict) -> None:
    """
    Log a concise cross-module summary for paper reporting.

    Covers: per-module significance counts, key findings per domain,
    and null-result interpretation.
    """
    section("STEP 5 — EXECUTIVE SUMMARY")

    section("Results by module (K=2)")
    logger.info(
        "  %-12s %6s  %7s  %7s  Description",
        "Module", "Total", "q<0.05", "q<0.10",
    )
    logger.info("  " + "-" * 75)
    for mod, cfg in MODULES.items():
        df_mod = dfs_mod.get(mod, pd.DataFrame())
        if df_mod.empty:
            continue
        n_tot = len(df_mod)
        n_05  = int((df_mod["q_fdr"] < FDR_ALPHA).sum())
        n_10  = int((df_mod["q_fdr"] < FDR_EXPLORE).sum())
        logger.info("  %-12s %6d  %7d  %7d  %s", mod, n_tot, n_05, n_10, cfg["description"])

    section("Key findings for paper")

    # ICFES
    df_icfes = dfs_mod.get("ICFES", pd.DataFrame())
    if not df_icfes.empty:
        sig = df_icfes[df_icfes["q_fdr"] < FDR_ALPHA].sort_values("q_fdr")
        logger.info("\n  ICFES — %d variables q<0.05", len(sig))
        logger.info("  -> GO-1 High has higher education and better ICFES scores than GO-2 Low.")
        logger.info("  -> Convergent validity of cognitive clustering:")
        logger.info("     WASI+CVLT+VMI phenotypes predict national exam performance.")
        logger.info("  -> Ref: Bhojwani 2019, Johnson 2016.")
        if len(sig) > 0:
            top = sig.iloc[0]
            delta_str = "{:.1f}".format(top["delta_pct_go2_vs_go1"]) if not np.isnan(top["delta_pct_go2_vs_go1"]) else "n/a"
            logger.info(
                "\n  Top variable: %s | GO-1=%.2f  GO-2=%.2f  q=%.5f  delta%%=%s",
                top["label"], top["mean_go1"], top["mean_go2"], top["q_fdr"], delta_str,
            )

    # ABCL / CONNERS
    for mod, name, ref in [
        ("ABCL",    "Adult behaviour",
         "Hack 2009 (behavioural problems in preterm adults)"),
        ("CONNERS", "Attention/ADHD symptoms",
         "Johnson 2010 (ADHD in preterm adults)"),
    ]:
        df_mod = dfs_mod.get(mod, pd.DataFrame())
        if df_mod.empty:
            continue
        sig = df_mod[df_mod["q_fdr"] < FDR_EXPLORE]
        n_sig = len(sig)
        logger.info("\n  %s — %s: %d vars q<0.10", mod, name, n_sig)
        if n_sig == 0:
            logger.info("  -> No significant differences between GO-1 and GO-2.")
            logger.info("  -> Cognitive phenotypes do not translate into behaviour/attention differences.")
        else:
            logger.info("  -> Ref: %s", ref)

    # Null modules
    null_mods = [
        m for m in MODULES
        if not dfs_mod.get(m, pd.DataFrame()).empty
        and int((dfs_mod[m]["q_fdr"] < FDR_EXPLORE).sum()) == 0
    ]
    if null_mods:
        logger.info("\n  Modules with no findings (q<0.10): %s", null_mods)
        logger.info("  -> GO-i is not associated with metabolic profile, vision,")
        logger.info("     hearing, or life habits in this cohort.")
        logger.info("  -> Important result: GO-i reflects cognition, not general health.")


executive_summary(df_all, dfs_mod)


## 10. Step 6 — MLflow Experiment Tracking

Logs all parameters, per-module significance counts, CSV outputs,
and all inline figures as a single MLflow run.

> **Databricks note:** tracking server is pre-configured — no URI setup needed.

In [0]:
def log_to_mlflow(
    df_all, df_miss, df_outlier_report,
    fig_missing, fig_outliers, fig_list,
):
    """
    Log the complete Part C run to MLflow.

    Logged items
    ------------
    params   : pipeline configuration (thresholds, module list, test types)
    metrics  : per-module q<0.05 and q<0.10 counts + totals
    artifacts:
        - partc_results_full.csv         — all variable results
        - partc_results_significant.csv  — q<0.10 subset
        - partc_missing_report.csv
        - partc_outlier_report.csv
        - all inline figures (saved to temp, logged, not persisted to disk)
    """
    section("STEP 6 — MLFLOW LOGGING")
    mlflow.set_registry_uri("databricks-uc") # Use "databricks" instead if you are not using Unity Catalog
    mlflow.pyspark.ml.autolog(disable=True)
    mlflow.set_experiment(f"/Users/ja.gutierrezb@uniandes.edu.co/pasoc_vars20a_v4")

    with mlflow.start_run(run_name="partc_vars20y_v3") as run:

        # ── Parameters ────────────────────────────────────────────────
        mlflow.log_params({
            "fdr_primary":      FDR_ALPHA,
            "fdr_exploratory":  FDR_EXPLORE,
            "fdr_method":       "Benjamini-Hochberg_per_module",
            "min_n_group":      MIN_N_GROUP,
            "iqr_mult":         IQR_MULT,
            "winsorize":        WINSORIZE,
            "n_modules":        len(MODULES),
            "modules":          ",".join(MODULES.keys()),
            "test_continuous":  "MannWhitneyU_twosided + Spearman_r",
            "test_categorical":  "Pearson_chi2",
            "n_vars_total":     int(len(df_all)),
        })

        # ── Metrics: per-module significance counts ────────────────────
        metrics = {}
        for mod in MODULES:
            sub_m = df_all[df_all["module"] == mod]
            if sub_m.empty:
                continue
            key_05 = "sig_q05_" + mod.lower()
            key_10 = "sig_q10_" + mod.lower()
            key_n  = "n_vars_" + mod.lower()
            metrics[key_05] = int((sub_m["q_fdr"] < FDR_ALPHA).sum())
            metrics[key_10] = int((sub_m["q_fdr"] < FDR_EXPLORE).sum())
            metrics[key_n]  = len(sub_m)
        metrics["total_sig_q05"] = int((df_all["q_fdr"] < FDR_ALPHA).sum())
        metrics["total_sig_q10"] = int((df_all["q_fdr"] < FDR_EXPLORE).sum())
        metrics["total_vars"]    = int(len(df_all))
        mlflow.log_metrics(metrics)

        # ── CSV and figure artifacts ───────────────────────────────────
        with tempfile.TemporaryDirectory() as tmp:

            # Results CSVs
            df_sig = df_all[df_all["q_fdr"] < FDR_EXPLORE].sort_values(["module", "q_fdr"])
            csv_pairs = [
                (df_all,           OUT_FULL),
                (df_sig,           OUT_SIG),
                (df_miss,          OUT_MISSING),
                (df_outlier_report, OUT_OUTLIER),
            ]
            for df_csv, fname in csv_pairs:
                if df_csv is not None and not df_csv.empty:
                    path = os.path.join(tmp, fname)
                    df_csv.to_csv(path, index=False)
                    mlflow.log_artifact(path, artifact_path="outputs")

            # Figures — save to temp, log; display already happened inline
            all_figs = []
            if fig_missing is not None:
                all_figs.append((fig_missing, "fig_01_missing.png"))
            if fig_outliers is not None:
                all_figs.append((fig_outliers, "fig_02_outliers.png"))
            all_figs.extend(fig_list)

            for fig, fname in all_figs:
                path = os.path.join(tmp, fname)
                fig.savefig(path, dpi=150, bbox_inches="tight")
                mlflow.log_artifact(path, artifact_path="figures")

            logger.info("All artifacts logged to MLflow ✓")

        run_id = run.info.run_id
        logger.info("\nMLflow run ID  : %s", run_id)
        logger.info("Experiment     : %s", MLFLOW_EXPERIMENT)

    return run_id


run_id = log_to_mlflow(
    df_all, df_miss, df_outlier_report,
    fig_missing, fig_outliers, fig_list,
)
print("\n✓ MLflow run completed. Run ID: " + run_id)


## 11. Run Summary

In [0]:
section("RUN SUMMARY — Part C vars 20y v3")
n_sig = int((df_all["q_fdr"] < FDR_ALPHA).sum())
n_exp = int((df_all["q_fdr"] < FDR_EXPLORE).sum())
logger.info("  Total variables analysed : %d", len(df_all))
logger.info("  Significant q<0.05       : %d", n_sig)
logger.info("  Exploratory  q<0.10      : %d", n_exp)
logger.info("  Test used                : Mann-Whitney U (two-sided, K=2)")
logger.info("  FDR correction           : Benjamini-Hochberg per module")
logger.info("  Winsorization            : +-%s xIQR (continuous only)", str(IQR_MULT))
logger.info("  MLflow run ID            : %s", run_id)
logger.info("")
logger.info("  Next step: Part D — Incremental predictive model")
